This version incorporates the followings:

1. P2 uses a pure best response
2. Compute the exact minimax loss for small I (P1 types) and K (time steps)
3. Both players use time-specific closed-loop policy networks 

In [182]:
# ==============================================================
#  setup.py
#  --------------------------------------------------------------
#  Global imports, device configuration, physical / game constants,
#  and hyper-parameters for closed-loop neural policies.
# ==============================================================

import math, random, os, sys, copy, pathlib, itertools
from typing import Dict, Tuple, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

# -----------------------------------------------------------------
#  Device switch (CPU for quick debugging, GPU for full training)
# -----------------------------------------------------------------
USE_GPU   = torch.cuda.is_available()
device    = torch.device("cuda" if USE_GPU else "cpu")
print("Running on:", device)

# -----------------------------------------------------------------
#  Game constants
# -----------------------------------------------------------------
τ         = 1.0 / 10.0                  # time step (s)
T         = 1.0                       # horizon   (s)
K         = int(T / τ)                # number of discrete steps
I         = 2                         # number of target types  (=|Θ|)

BOX_POS   = 2.0                       # |position|
BOX_VEL   = 30.0                       # |velocity|
BOX_ACC   = 8.0                       # |accel|

Z_TARGETS = torch.tensor([[0.0,  1.0, 0.0, 0.0],
                          [0.0, -1.0, 0.0, 0.0]],
                         device=device)                # shape (I,4)

Kmat      = torch.diag(torch.tensor([1., 1., 0., 0.], device=device))
R1        = torch.diag(torch.tensor([0.05, 0.025], device=device))
R2        = torch.diag(torch.tensor([0.05, 0.100], device=device))

# -----------------------------------------------------------------
#  Observation feature dimension
#     • t  (1)
#     • joint state  x  (8)
#     • public belief  p  (I-1)  – simplex barycentric coord
# -----------------------------------------------------------------
BELIEF_DIM = I - 1
FEAT_DIM   = 8 + BELIEF_DIM        # = 9  when I = 2 (state 8  +  belief, networks are time specific)

# -----------------------------------------------------------------
#  Hyper-parameters
# -----------------------------------------------------------------
LR_P1        = 3e-2                    # learning rate – player-1 network
LR_P2        = 1e-1                    # learning rate – player-2 network
BATCH_SIZE   = 32
EPOCHS       = 12000
MOMENTUM     = 0.6                     # β in DS-GDA
C_SQUARE_P1  = 10.0                    # momentum-clip radius²  (player-1)
C_SQUARE_P2  = 10.0                    # momentum-clip radius²  (player-2)

SEED         = 2341
TEMPERATURE  = 1.0                     # row-softmax temperature for P1
NOISE_SCALE  = -10.0                   # exploration σ (log-std)  (unused by default)
HIDDEN_DIM   = 32


torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# -----------------------------------------------------------------
#  Utility – belief coordinate  Δᴵ → ℝᴵ⁻¹
# -----------------------------------------------------------------
def belief_coord(p: Tensor) -> Tensor:
    """
    Convert public belief p ∈ Δᴵ to its I-1 free barycentric coordinates.
    For I = 2 this is simply p₀.
    """
    return p[..., :BELIEF_DIM]

Running on: cpu


In [183]:
# =============================================================
#  diff_env.py
#  -------------------------------------------------------------
#  Differentiable rollout of Hexner’s game  (closed-loop version)
#
#  Usage
#  -----
#      env   = HexnerDiffEnv(batch_size=512)
#      loss, traj = env.rollout(p1_net, p2_net)
#      loss.backward()                     # hits both networks
#
#  Notes
#  -----
#  • Works for *any* Player-1 implementation that exposes
#        u1 , misc = p1_net.action_only(obs , i_star)
#    where
#        misc["A"] : (B,I,I)  – row-normalised probabilities
#        misc["j"] : (B,)     – chosen discrete index (no-grad)
#        misc["μ"] : (B,I,I,2)  prototype action table (optional,
#                               logged for visualisation).
#  • Player-2 only needs .action_only(obs) →  (u2 , misc).
# =============================================================
from typing import List
import torch
from torch import Tensor

# from setup import (device, τ, K, I,
#                    BOX_POS, BOX_VEL, BOX_ACC,
#                    Z_TARGETS, Kmat, R1, R2)

# -------------------------------------------------------------------------
#  Prior on the hidden type
# -------------------------------------------------------------------------
P0 = torch.full((1, I), 1.0 / I, device=device)   # uniform prior


class HexnerDiffEnv:
    """
    Differentiable environment for Hexner's two-player zero-sum game.

    All internal tensors live on `device`; *no* @torch.no_grad anywhere, so
    gradients flow through dynamics, Bayes update, and time counters.
    """

    # ---------------------------------------------------------------------
    def __init__(self, batch_size: int = 1):
        self.B = batch_size
        self.reset()

    # ---------------------------------------------------------------------
    #  Cost functions
    # ---------------------------------------------------------------------
    def _running_loss(self, u1: Tensor, u2: Tensor) -> Tensor:
        """
        Instantaneous cost   ℓ(u₁,u₂)·τ     –  positive for P1, negative for P2
        Returns shape (B,)
        """
        u1_cost = (u1.unsqueeze(1) @ R1 @ u1.unsqueeze(-1)).squeeze(-1).squeeze(-1)
        u2_cost = (u2.unsqueeze(1) @ R2 @ u2.unsqueeze(-1)).squeeze(-1).squeeze(-1)
        return 0.5 * (u1_cost - u2_cost) * τ

    def _terminal_loss(self) -> Tensor:
        """
        g_i(x_K)   – final positional advantage of P1 over P2.
        Returns shape (B,)
        """
        x1 = self.x[:, 0:4]
        x2 = self.x[:, 4:8]
        del1 = x1 - Z_TARGETS[self.i_star]         # (B,4)
        del2 = x2 - Z_TARGETS[self.i_star]
        term = 0.5 * ((del1 @ Kmat * del1).sum(-1) -
                      (del2 @ Kmat * del2).sum(-1))
        return term

    # ---------------------------------------------------------------------
    #  Physics – second-order point-mass integrator
    # ---------------------------------------------------------------------
    def step_dynamics(self, u1: Tensor, u2: Tensor):
        """
        Semi-implicit Euler on (pos,vel) with saturation boxes.
            p ← p + v·τ + ½ a·τ²
            v ← v + a·τ
        """
        # --- NEW: guarantee shapes (B,2) to avoid accidental (B,1,2) ---
        u1 = u1.reshape(self.B, 2)
        u2 = u2.reshape(self.B, 2)
        # ----------------------------------------------------------------

        pos1, vel1 = self.x[:, 0:2], self.x[:, 2:4]
        pos2, vel2 = self.x[:, 4:6], self.x[:, 6:8]

        pos1_new = (pos1 + vel1 * τ + 0.5 * u1 * τ ** 2).clamp(-BOX_POS, BOX_POS)
        vel1_new = (vel1 + u1 * τ).clamp(-BOX_VEL, BOX_VEL)
        pos2_new = (pos2 + vel2 * τ + 0.5 * u2 * τ ** 2).clamp(-BOX_POS, BOX_POS)
        vel2_new = (vel2 + u2 * τ).clamp(-BOX_VEL, BOX_VEL)

        self.x = torch.cat([pos1_new, vel1_new, pos2_new, vel2_new], dim=-1)
        self.t = self.t + τ

    # ---------------------------------------------------------------------
    #  Belief update  –  differentiable Bayes rule
    # ---------------------------------------------------------------------
    @staticmethod
    def _bayes_update(p: Tensor, A: Tensor, j: Tensor) -> Tensor:
        """
        Public belief update  p_{t+1} ∝ A[:,j] ⊙ p.
        Parameters
        ----------
        p : (B,I)              – current belief
        A : (B,I,I)            – row-normalised matrix from P1
        j : (B,)  (no grad)    – sampled column index
        Returns
        -------
        p_next : (B,I)         – updated belief  (∑ = 1)
        """
        B, I = p.shape
        j = j.to(dtype=torch.long).view(B, 1, 1)          # (B,1,1)

        # Gather the selected column for every batch element --------------
        # index  shape : (B, I, 1) – replicate the batch-specific j along row dim
        index = j.expand(-1, I, 1)                        # (B,I,1)
        Aj = A.gather(dim=2, index=index).squeeze(-1)     # (B,I)

        # Bayes rule ------------------------------------------------------
        numer = Aj * p                                    # (B,I)
        denom = numer.sum(-1, keepdim=True).clamp_min(1e-8)
        return numer / denom                              # (B,I)

    # ---------------------------------------------------------------------
    #  Main rollout
    # ---------------------------------------------------------------------
    def rollout(self, p1_net, p2_net):
        """
        Unroll K steps and return:
            • total_loss  – scalar differentiable
            • traj        – list of diagnostics per step  (for viz)
        """
        running_costs: List[Tensor] = []

        for k in range(K):
            # ------------- build observation ----------------------------
            obs = {"t": self.t, "x": self.x, "p": self.p}

            # ------------- query policies -------------------------------
            u1, misc1 = p1_net.action_only(obs, self.i_star, k)   # (B,2)
            u2, misc2 = p2_net.action_only(obs, k)                # (B,2)

            # ------------- physics + cost -------------------------------
            self.step_dynamics(u1, u2)
            running_costs.append(self._running_loss(u1, u2))

            # ------------- Bayes belief update --------------------------
            self.p = self._bayes_update(self.p, misc1["A"], misc1["j"])

            # ------------- trajectory logging (for visualisation) -------
            self.traj.append({
                "t"      : self.t.detach().cpu(),
                "p1_xy"  : self.x[:, 0:2].detach().cpu(),
                "p2_xy"  : self.x[:, 4:6].detach().cpu(),
                "belief" : self.p.detach().cpu(),
                "A"      : misc1["A"].cpu(),        # (B,I,I)
                "μ"      : misc1["μ"].cpu(),        # (B,I,2)
                "j"      : misc1["j"].cpu()
            })

        total_running = torch.stack(running_costs, dim=0).sum(0)      # (B,)
        total_loss    = (total_running + self._terminal_loss()).mean()  # scalar
        return total_loss, self.traj

    # ---------------------------------------------------------------------
    def reset(self):
        """
        Re-initialise type, state, belief, timer, and trajectory buffer.
        Returns the initial public observation (no grad) for convenience.
        """
        # 1) sample hidden type  i_star  and initialise public belief
        self.i_star = torch.multinomial(P0, self.B, replacement=True)  # (B,)
        self.p      = P0.repeat(self.B, 1).clone()                     # (B,I)

        # 2) state & time
        self.x = torch.zeros(self.B, 8, device=device)
        self.x[:, 0] = -0.5        # P1 initial x-coord
        self.x[:, 4] = +0.5        # P2 initial x-coord
        self.t = torch.zeros(self.B, device=device)

        # 3) clear stored trajectory
        self.traj: list[dict] = []

        # 4) return observation
        return {"t": self.t.detach(),
                "x": self.x.detach(),
                "p": self.p.detach()}

In [184]:
# =============================================================
#  networks_closedloop.py
#  -------------------------------------------------------------
#  • P1TimeIndexed –  K independent sub-nets, one per time-step.
#                     Each sub-net outputs:
#                        A_logits : (B,I,I)
#                        μ        : (B,I,2)   (global prototypes)
#  • P2BestResponse – unchanged.
# =============================================================
import torch, torch.nn as nn
from torch import Tensor
# from setup import (I, K, FEAT_DIM, BOX_ACC, belief_coord,
                #    TEMPERATURE, device)

# -------------------------------------------------------------
class _SingleStepNet(nn.Module):
    """One step-specific policy π₁ᵏ."""
    def __init__(self, hidden: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(FEAT_DIM, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU()
        )
        self.logit_head = nn.Linear(hidden, I * I)   # (I×I) logits
        self.mu_head    = nn.Linear(hidden, I * 2)   # I prototypes

    def forward(self, obs: dict[str, Tensor]) -> dict[str, Tensor]:
        h = self.net(torch.cat([obs["x"],
                                belief_coord(obs["p"])], dim=-1))
        A_logits = self.logit_head(h).view(-1, I, I)      # (B,I,I)
        μ        = BOX_ACC * torch.tanh(
                       self.mu_head(h).view(-1, I, 2))    # (B,I,2)
        return {"A_logits": A_logits, "μ": μ}

# -------------------------------------------------------------
class P1TimeIndexed(nn.Module):
    """
    Wrapper holding K independent sub-nets π₁ᵏ.
    API:
        u , misc = policy.action_only(obs, i_star, k)
    """
    def __init__(self, hidden: int = HIDDEN_DIM):
        super().__init__()
        self.subnets = nn.ModuleList([_SingleStepNet(hidden) for _ in range(K)])

    def forward(self, obs: dict[str, Tensor], k: int):
        return self.subnets[k](obs)

    @torch.no_grad()
    def action_only(self, obs: dict[str, Tensor], i_star: Tensor, k: int):
        """
        Select row i_star and argmax column for step k.
        """
        out        = self.forward(obs, k)
        A_logits   = out["A_logits"]                    # (B,I,I)
        μ_proto    = out["μ"]                           # (B,I,2)

        B_idx      = torch.arange(A_logits.size(0), device=A_logits.device)
        logits_row = A_logits[B_idx, i_star]
        j          = torch.argmax(logits_row, dim=-1)   # (B,)

        u          = μ_proto[B_idx, j]                  # (B,2)

        misc = {
            "A":   torch.softmax(A_logits / TEMPERATURE, dim=-1),
            "row": torch.softmax(logits_row, dim=-1),
            "j":   j,
            "μ":   μ_proto
        }
        return u, misc
    

# -------------------------------------------------------------
class _SingleStepP2(nn.Module):
    """One deterministic best-response sub-net for step k."""
    def __init__(self, hidden: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(FEAT_DIM, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU(),
            nn.Linear(hidden, 2), nn.Tanh()
        )
    def forward(self, obs):
        # from setup import BOX_ACC
        z = torch.cat([obs["x"], belief_coord(obs["p"])], dim=-1)  # no t
        return self.net(z) * BOX_ACC

# -------------------------------------------------------------
class P2TimeIndexed(nn.Module):
    """Wrapper with K independent best-response sub-nets."""
    def __init__(self, hidden: int = HIDDEN_DIM):
        super().__init__()
        self.subnets = nn.ModuleList([_SingleStepP2(hidden) for _ in range(K)])
    @torch.enable_grad()
    def forward(self, obs, k: int):
        return self.subnets[k](obs)                  # (B,2)
    def action_only(self, obs, k: int):
        u = self.forward(obs, k)
        return u, {"μ": u.detach()}

In [185]:
# =============================================================
#  exact_loss_closedloop.py
#  -------------------------------------------------------------
#  Exact game value  L(θ,φ)  for closed-loop Player 1.
#
#  Works for any  I×I  output P1 network that obeys
#        out = p1_net.forward(obs)
#        A_logits = out["A_logits"]   (B,I,I)
#        μ        = out["μ"]          (B,I,I,2)
#
#  The routine below enumerates **every** joint discrete
#  action sequence  j₀…j_{K−1} ∈ {0,…,I−1}ᴷ  (S = Iᴷ),
#  rolls out the dynamics deterministically under Player 2’s
#  best-response, and weights by the exact path probability.
#
#  For I = 2, K = 10 ⇒ S = 1024  (tractable).
# =============================================================
from __future__ import annotations
import itertools, torch
from torch import Tensor

# from setup import device, I, K, τ, P0
# from diff_env import HexnerDiffEnv

# ---------------------------------------------------------------------------
#  Helper – generate and memoise all sequences  (S,K)  long tensor
# ---------------------------------------------------------------------------
_SEQ_CACHE = {}
def _sequences(K: int, I: int = I) -> Tensor:
    if K not in _SEQ_CACHE:
        _SEQ_CACHE[K] = torch.tensor(
            list(itertools.product(range(I), repeat=K)),
            device=device, dtype=torch.long
        )                             # (S,K)
    return _SEQ_CACHE[K]

# ---------------------------------------------------------------------------
def exact_loss(p1_net, p2_net) -> Tensor:
    """
    Vectorised exact loss:
        • batch over all paths  S = I**K   (1024 for I=2,K=10)
        • two forward passes (one per type) instead of S.
    """
    seq = _sequences(K)            # (S,K)
    S   = seq.size(0)
    total = torch.tensor(0., device=device)

    for i_star in range(I):
        prior_i = P0[0, i_star]

        # ----- create one big batched environment ------------------------
        env = HexnerDiffEnv(batch_size=S)
        env.i_star.fill_(i_star)
        env.p.copy_(P0.repeat(S, 1))

        path_prob   = torch.ones(S, device=device)     # (S,)
        running_acc = torch.zeros(S, device=device)    # (S,)

        for k in range(K):
            j_k = seq[:, k]                           # (S,)

            obs = {"t": env.t, "x": env.x, "p": env.p}

            # P1 network for step k  (batched)
            out       = p1_net.forward(obs, k)
            A_logits  = out["A_logits"]               # (S,I,I)
            μ_proto   = out["μ"]                      # (S,I,2)

            # soft row probs and multiply into path_prob
            row_probs = torch.softmax(A_logits[:, i_star], dim=-1)  # (S,I)
            prob_j    = row_probs.gather(1, j_k.unsqueeze(1)).squeeze(1)
            path_prob = path_prob * prob_j

            # P1 & P2 continuous controls
            u1 = μ_proto[torch.arange(S, device=device), j_k]       # (S,2)
            u2, _ = p2_net.action_only(obs, k)                      # (S,2)

            # dynamics + running cost
            env.step_dynamics(u1, u2)
            running_acc = running_acc + env._running_loss(u1, u2)

            # Bayes update (vectorised)
            A_soft = torch.softmax(A_logits, dim=-1)
            env.p  = env._bayes_update(env.p, A_soft, j_k)

        # terminal cost
        L_paths = running_acc + env._terminal_loss()    # (S,)

        total += prior_i * (path_prob * L_paths).sum()

    return total

In [186]:
# =============================================================
#  dsgda_closedloop.py
#  -------------------------------------------------------------
#  Deterministic Simultaneous Gradient Descent–Ascent (DS-GDA)
#  with **exact** gradients for the closed-loop formulation.
#
#  Player-1  : neural network P1ClosedLoop (all parameters trainable)
#  Player-2  : deterministic best-response network P2BestResponse
#
#  The update rule follows the original paper:
#      m ← β·m + (1-β)·∇
#      m ← clip₂(m, C)           (vector norm ≤ C)
#      θ ← θ  +  α·m      (ascent   for P1)
#      φ ← φ  −  α·m      (descent  for P2)
#
#  Usage
#  -----
#      solver = DSGDA_ClosedLoop(K=K,
#                                lr_p1=LR_P1, lr_p2=LR_P2,
#                                beta=MOMENTUM,
#                                C2_p1=C_SQUARE_P1,
#                                C2_p2=C_SQUARE_P2)
#      for epoch in range(EPOCHS):
#          stats = solver.step()
# =============================================================
from __future__ import annotations
import math, itertools, torch
from torch import Tensor

# from setup import (device, MOMENTUM, C_SQUARE_P1, C_SQUARE_P2,
#                    LR_P1, LR_P2)

# from networks_closedloop import P1ClosedLoop, P2BestResponse
# from exact_loss_closedloop import exact_loss


# ---------------------------------------------------------------------------
#  Momentum buffer with in-place clipping
# ---------------------------------------------------------------------------
class MomentumBuffer:
    def __init__(self, params, beta: float):
        self.params = list(params)
        self.m      = [torch.zeros_like(p, device=device) for p in self.params]
        self.beta   = beta

    def update(self):
        """Exponential moving average of the current gradients."""
        for m, p in zip(self.m, self.params):
            if p.grad is not None:
                m.mul_(self.beta).add_(p.grad, alpha=1.0 - self.beta)

    def clip_(self, C: float):
        """ℓ₂ clip each momentum vector to length ≤ C."""
        for m in self.m:
            n = m.norm()
            if n > C:
                m.mul_(C / n)

    def apply_step(self, ascent: bool, lr: float):
        """Gradient *ascent* if ascent=True, else descent."""
        sign = +1.0 if ascent else -1.0
        for p, m in zip(self.params, self.m):
            p.data.add_(m, alpha=sign * lr)


# ---------------------------------------------------------------------------
#  DS-GDA driver
# ---------------------------------------------------------------------------
class DSGDA_ClosedLoop:
    def __init__(self,
                 K: int,
                 lr_p1: float = LR_P1,
                 lr_p2: float = LR_P2,
                 beta: float  = MOMENTUM,
                 C2_p1: float = C_SQUARE_P1,
                 C2_p2: float = C_SQUARE_P2,
                 hidden: int  = HIDDEN_DIM):
        """
        Parameters
        ----------
        K        : planning horizon (# discrete steps) – only stored for logs
        lr_p1    : learning rate for Player-1 parameters
        lr_p2    : learning rate for Player-2 parameters
        beta     : momentum coefficient (0.6 ≈ recommended by paper)
        C2_p1    : square of clipping radius for P1 momentum
        C2_p2    : square of clipping radius for P2 momentum
        hidden   : shared hidden width for both networks
        """
        self.K        = K
        self.lr_p1    = lr_p1
        self.lr_p2    = lr_p2
        self.C1       = math.sqrt(C2_p1)
        self.C2       = math.sqrt(C2_p2)

        # networks --------------------------------------------------------
        self.p1 = P1TimeIndexed(hidden=hidden).to(device)
        self.p2 = P2TimeIndexed(hidden=hidden).to(device)

        # parameter lists (skip frozen tensors)
        self.p1_vars = [p for p in self.p1.parameters() if p.requires_grad]
        self.p2_vars = [p for p in self.p2.parameters() if p.requires_grad]

        # momentum buffers
        self.buf_p1  = MomentumBuffer(self.p1_vars, beta)
        self.buf_p2  = MomentumBuffer(self.p2_vars, beta)

    # ---------------------------------------------------------------------
    def step(self) -> dict[str, float]:
        """
        Perform one GDA iteration and return diagnostics.
        """
        # ---------- zero old grads ---------------------------------------
        for p in itertools.chain(self.p1_vars, self.p2_vars):
            if p.grad is not None:
                p.grad.zero_()

        # ---------- exact objective & back-prop --------------------------
        loss = exact_loss(self.p1, self.p2)      # scalar (P1 maximises)
        loss.backward()

        # ---------- EMA momentum update & ℓ₂ clip ------------------------
        self.buf_p1.update();  self.buf_p2.update()
        self.buf_p1.clip_(self.C1); self.buf_p2.clip_(self.C2)

        # ---------- gradient (a/des)cent step ----------------------------
        self.buf_p1.apply_step(ascent=False,  lr=self.lr_p1)   # maximise
        self.buf_p2.apply_step(ascent=True, lr=self.lr_p2)   # minimise

        # ---------- diagnostics -----------------------------------------
        g_p1 = torch.stack([m.norm() for m in self.buf_p1.m]).mean().item()
        g_p2 = torch.stack([m.norm() for m in self.buf_p2.m]).mean().item()

        return {"L": loss.item(),
                "g_p1": g_p1,
                "g_p2": g_p2}

In [187]:
# =============================================================
#  visualize.py
#  -------------------------------------------------------------
#  Roll-out visualisation with full I×I probability & action
#  tables shown at every time-step.
# =============================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

import torch

# from setup import (I, K, τ, T, BOX_POS, Z_TARGETS)
# from diff_env import HexnerDiffEnv


# ---------------------------------------------------------------------------
def animate_episode(p1_net, p2_net, fps: int = 5) -> HTML:
    """
    Roll out one episode (batch = 1) using the **current** policies and
    display:
        • console prints of the full  I×I  probability matrix  A
          and prototype action table  μ  at every step;
        • an animation of the planar trajectories plus belief trace.
    """
    env = HexnerDiffEnv(batch_size=1)
    obs = env.reset()
    i_star = env.i_star                         # tensor of shape (1,)

    # buffers for the animation ------------------------------------------------
    p_traj, t_traj, p1_xy, p2_xy = [], [], [], []

    print("\n========== DEBUG ROLL-OUT ==========")
    for step in range(K):
        # ---------- policy queries ------------------------------------------
        with torch.no_grad():
            u1, misc1 = p1_net.action_only(obs, i_star, step)
            u2, _     = p2_net.action_only(obs, step)

        # ---------- diagnostics ---------------------------------------------
        print(f"\n[t = {step*τ: .2f} s]")
        # full probability matrix A  (I×I)
        A_mat = misc1["A"][0].cpu().detach().numpy()           # (I,I)
        print("P1 probability matrix  A :")
        for r in range(I):
            print(f"  row {r}:", np.round(A_mat[r], 3))

        # prototype action table μ  (shared across rows)
        mu_tbl = misc1["μ"][0].cpu().numpy()              # (I,2)
        print("Global prototype actions μ :")
        for idx, vec in enumerate(mu_tbl):
            print(f"  idx {idx}: {np.round(vec, 3)}")

        j = misc1["j"].item()
        print("Chosen prototype idx :", j)
        print("u₁ action       :", u1.squeeze(0).cpu().detach().numpy())
        print("u₂ action       :", u2.squeeze(0).cpu().detach().numpy())

        # ---------- advance environment -------------------------------------
        env.step_dynamics(u1, u2)
        env.p = env._bayes_update(env.p, misc1["A"], misc1["j"])

        # ---------- update obs & buffers ------------------------------------
        obs = {"t": env.t.detach(), "x": env.x.detach(), "p": env.p.detach()}

        p1_xy.append(env.x[0, 0:2].cpu().detach().numpy())
        p2_xy.append(env.x[0, 4:6].cpu().detach().numpy())
        p_traj.append(env.p[0, 0].item())             # belief of type-0
        t_traj.append((step + 1) * τ)

    print("====================================\n")

    # ------------------------------------------------------------------------
    #  Build the Matplotlib animation
    # ------------------------------------------------------------------------
    p_belief = np.array([0.5] + p_traj)               # prepend t=0 value
    times    = np.array([0.0] + t_traj)
    p1_xy    = np.vstack([env.x[0, 0:2].cpu().detach().numpy()] + p1_xy)
    p2_xy    = np.vstack([env.x[0, 4:6].cpu().detach().numpy()] + p2_xy)

    fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(5, 8),
                                   gridspec_kw={"height_ratios": [3, 1]})
    # ----- top: trajectories ----------------------------------------------
    ax0.set_xlim(-BOX_POS - .2, BOX_POS + .2)
    ax0.set_ylim(-BOX_POS - .2, BOX_POS + .2)
    ax0.set_aspect("equal")
    ax0.set_title("Hexner – trajectories")

    # target markers
    for idx, tgt in enumerate(Z_TARGETS.cpu().numpy()):
        star_kw = dict(marker="*", ms=14,
                       color="black" if idx == i_star.item() else "grey")
        ax0.plot(tgt[0], tgt[1], **star_kw)

    p1_sc = ax0.scatter([], [], s=80, c="red")
    p2_sc = ax0.scatter([], [], s=80, c="blue")

    # ----- bottom: belief ---------------------------------------------------
    ax1.set_xlim(0, T)
    ax1.set_ylim(-.05, 1.05)
    ax1.set_xlabel("time (s)")
    ax1.set_ylabel("public belief p[type-0]")
    ax1.plot(times, p_belief, color="blue")
    ax1.axhline(y=float(i_star.item() == 0), color="black", ls="--")
    ax1.set_title("Belief trajectory")

    def init():
        p1_sc.set_offsets(np.empty((0, 2)))
        p2_sc.set_offsets(np.empty((0, 2)))
        return p1_sc, p2_sc

    def update(frame):
        p1_sc.set_offsets(p1_xy[frame])
        p2_sc.set_offsets(p2_xy[frame])
        return p1_sc, p2_sc

    ani = animation.FuncAnimation(fig, update, frames=len(times),
                                  init_func=init, blit=True,
                                  interval=1000 / fps)
    plt.close(fig)
    return HTML(ani.to_jshtml())

In [188]:
# -------------------------------------------------------------
#  Train with periodic visualisation
# -------------------------------------------------------------
# from dsgda_closedloop import DSGDA_ClosedLoop
# from visualize import animate_episode
from IPython.display import display

# instantiate solver
solver = DSGDA_ClosedLoop(
    K=K,
    lr_p1=LR_P1,
    lr_p2=LR_P2,
    beta=MOMENTUM,
    C2_p1=C_SQUARE_P1,
    C2_p2=C_SQUARE_P2,
)

VIS_EVERY = 5000          # epochs between animations

for epoch in range(EPOCHS):
    stats = solver.step()

    # console log
    if epoch % VIS_EVERY == 0:
        print(f"[{epoch:04d}]  "
              f"L = {stats['L']:+.3f}   "
              f"‖m₁‖ = {stats['g_p1']:.3f}   "
              f"‖m₂‖ = {stats['g_p2']:.3f}")

        # inline animation
        print("🎬  visual check")
        display(animate_episode(solver.p1, solver.p2))

[0000]  L = -0.129   ‖m₁‖ = 0.019   ‖m₂‖ = 0.050
🎬  visual check

========== DEBUG ROLL-OUT ==========

[t =  0.00 s]
P1 probability matrix  A :
  row 0: [0.439 0.561]
  row 1: [0.481 0.519]
Global prototype actions μ :
  idx 0: [-0.286  0.861]
  idx 1: [-0.189  0.771]
Chosen prototype idx : 1
u₁ action       : [[-0.18858908  0.77138436]]
u₂ action       : [0.46752375 0.9346239 ]

[t =  0.10 s]
P1 probability matrix  A :
  row 0: [0.528 0.472]
  row 1: [0.517 0.483]
Global prototype actions μ :
  idx 0: [-0.561  0.112]
  idx 1: [ 0.251 -1.017]
Chosen prototype idx : 0
u₁ action       : [[-0.56144047  0.11174539]]
u₂ action       : [-1.3188496  1.067928 ]

[t =  0.20 s]
P1 probability matrix  A :
  row 0: [0.462 0.538]
  row 1: [0.52 0.48]
Global prototype actions μ :
  idx 0: [0.578 1.093]
  idx 1: [ 0.019 -0.572]
Chosen prototype idx : 1
u₁ action       : [[ 0.01920366 -0.57168907]]
u₂ action       : [1.0774044  0.12882744]

[t =  0.30 s]
P1 probability matrix  A :
  row 0: [0.568 0.4

[5000]  L = -0.224   ‖m₁‖ = 0.000   ‖m₂‖ = 0.000
🎬  visual check

========== DEBUG ROLL-OUT ==========

[t =  0.00 s]
P1 probability matrix  A :
  row 0: [0.458 0.542]
  row 1: [0.458 0.542]
Global prototype actions μ :
  idx 0: [ 1.243e+00 -1.000e-03]
  idx 1: [ 1.243e+00 -1.000e-03]
Chosen prototype idx : 1
u₁ action       : [[ 1.2432358e+00 -5.1096082e-04]]
u₂ action       : [-1.2410899e+00 -1.2820959e-04]

[t =  0.10 s]
P1 probability matrix  A :
  row 0: [0.525 0.475]
  row 1: [0.525 0.475]
Global prototype actions μ :
  idx 0: [ 1.114 -0.   ]
  idx 1: [ 1.114 -0.   ]
Chosen prototype idx : 0
u₁ action       : [[ 1.1136687e+00 -2.8693676e-04]]
u₂ action       : [-1.1096413e+00  4.9889088e-05]

[t =  0.20 s]
P1 probability matrix  A :
  row 0: [0.488 0.512]
  row 1: [0.488 0.512]
Global prototype actions μ :
  idx 0: [ 0.984 -0.   ]
  idx 1: [0.984 0.   ]
Chosen prototype idx : 1
u₁ action       : [[9.8411089e-01 2.1135807e-04]]
u₂ action       : [-9.7818798e-01  2.1725893e-04]

[t

[10000]  L = -0.224   ‖m₁‖ = 0.000   ‖m₂‖ = 0.000
🎬  visual check

========== DEBUG ROLL-OUT ==========

[t =  0.00 s]
P1 probability matrix  A :
  row 0: [0.458 0.542]
  row 1: [0.458 0.542]
Global prototype actions μ :
  idx 0: [1.242 0.   ]
  idx 1: [1.242 0.   ]
Chosen prototype idx : 1
u₁ action       : [[1.2422601e+00 3.3062696e-04]]
u₂ action       : [-1.2416654e+00 -9.1090798e-05]

[t =  0.10 s]
P1 probability matrix  A :
  row 0: [0.525 0.475]
  row 1: [0.525 0.475]
Global prototype actions μ :
  idx 0: [1.112 0.   ]
  idx 1: [1.112 0.   ]
Chosen prototype idx : 0
u₁ action       : [[1.1119878e+00 4.7415495e-04]]
u₂ action       : [-1.1106622e+00 -4.0531158e-05]

[t =  0.20 s]
P1 probability matrix  A :
  row 0: [0.488 0.512]
  row 1: [0.488 0.512]
Global prototype actions μ :
  idx 0: [0.982 0.001]
  idx 1: [0.982 0.001]
Chosen prototype idx : 1
u₁ action       : [[9.817287e-01 6.147623e-04]]
u₂ action       : [-9.796592e-01  1.001358e-05]

[t =  0.30 s]
P1 probability matrix